# Random Forest y caja gris en Kaggle

Entrena modelos supervisados para retornos futuros del oro. Usa el contexto UMAP si fue generado por el notebook 01.

In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import importlib.util

def find_repo():
    candidates = [Path.cwd(), Path('/kaggle/input')]
    for root in candidates:
        if not root.exists():
            continue
        for config in root.rglob('configs/project.yaml'):
            repo = config.parents[1]
            if (repo / 'src' / 'gold_policy_analysis').exists():
                return repo
    return Path.cwd()

REPO_DIR = find_repo()
WORK_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else REPO_DIR
os.chdir(WORK_DIR)
sys.path.insert(0, str(REPO_DIR / 'src'))
CONFIG = REPO_DIR / 'configs' / 'project.yaml'
print('REPO_DIR =', REPO_DIR)
print('WORK_DIR =', WORK_DIR)
print('CONFIG =', CONFIG)

In [ ]:
packages = {
    'pandas': 'pandas',
    'openpyxl': 'openpyxl',
    'sklearn': 'scikit-learn',
    'matplotlib': 'matplotlib',
    'yaml': 'pyyaml',
}
missing = [pip_name for import_name, pip_name in packages.items() if importlib.util.find_spec(import_name) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
print('Dependencies ready')

In [ ]:
from pathlib import Path

umap_context = Path('data/processed/policy_umap_embedding.csv')
if not umap_context.exists():
    try:
        from gold_policy_analysis.pipeline import run_policy_map
        run_policy_map(CONFIG)
    except Exception as exc:
        print('UMAP context was not generated; training continues without it:', exc)
print('UMAP context exists:', umap_context.exists())

In [ ]:
from gold_policy_analysis.supervised import run_random_forest_grey_box

outputs = run_random_forest_grey_box(CONFIG)
outputs

In [ ]:
import pandas as pd

metrics = pd.read_csv(outputs['metrics'])
metrics.sort_values(['horizon', 'rmse'])

In [ ]:
from IPython.display import Image, display

for horizon in [1, 3, 6]:
    key = f'predictions_figure_h{horizon}'
    if key in outputs:
        display(Image(filename=str(outputs[key])))

In [ ]:
for horizon in [1, 3, 6]:
    importance_path = Path(f'reports/tables/random_forest_feature_importance_h{horizon}.csv')
    if importance_path.exists():
        print('\nHorizon', horizon)
        display(pd.read_csv(importance_path).head(15))